# Pendulum on an accelerating cart: nonlinear vs. linearized

A cart receives a sharp (rectangular) acceleration pulse; the pendulum hanging from it swings. We sweep the pulse size **a** over 12 values and compare the full nonlinear model with the small-angle linearization.

**Convention:** θ is measured from the *downward* vertical (θ = 0 is the bottom of the arc). The attached PDF uses θ = 0 upright, so the gravity term has the opposite sign here.

## Derivation (Lagrangian, same method as the PDF)

Pendulum centre of mass, a distance $l$ from the pivot on the cart:

$$x_p = x + l\sin\theta,\qquad y_p = -l\cos\theta$$

Kinetic energy (with inertia $I_{cm}$ about the pendulum's own COM):

$$T = \tfrac12 M\dot x^2 + \tfrac12 m(\dot x_p^2+\dot y_p^2) + \tfrac12 I_{cm}\dot\theta^2 = \tfrac12 (M+m)\dot x^2 + m l\cos\theta\,\dot\theta\dot x + \tfrac12 I\dot\theta^2$$

where $I = I_{cm} + m l^2$ (parallel-axis theorem). Potential energy: $V = -m g l\cos\theta$. Lagrangian: $\mathcal L = T - V$.

Lagrange's equation for θ, with viscous pivot damping $-b\dot\theta$:

$$I\ddot\theta + m l\cos\theta\,\ddot x + m g l\sin\theta + b\dot\theta = 0 \quad\text{(nonlinear)}$$

Linearizing ($\sin\theta\approx\theta$, $\cos\theta\approx1$):

$$I\ddot\theta + b\dot\theta + m g l\,\theta = -m l\,\ddot x \quad\text{(linear)}$$

Natural frequency $\omega_0=\sqrt{mgl/I}$, damping ratio $\zeta = b/(2I\omega_0)$.

The cart acceleration $\ddot x(t)$ is prescribed, so only the θ equation is needed.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\TAtte\.anaconda\Anaconda.setup\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\TAtte\.anaconda\Anaconda.setup\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\TAtte\.anaconda\Anaconda.setup\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\TAtte\.anaconda\Anaconda.setup\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## 1. Parameters 

In [ ]:
g    = 9.81             # gravity [m/s^2]                     -> LOOKED UP
m    = 0.10             # pendulum mass [kg]                  -> PLACEHOLDER (measure with a scale)
L    = 0.50             # pendulum length [m]                 -> PLACEHOLDER (measure with a ruler)
l    = L / 2            # pivot-to-COM distance [m]           -> COMPUTED (uniform rod => L/2)
I_cm = m * L**2 / 12    # inertia about own COM [kg m^2]      -> COMPUTED (uniform rod formula)
I    = I_cm + m * l**2  # inertia about pivot [kg m^2]        -> COMPUTED (parallel-axis theorem)
b    = 0.0024            # pivot viscous damping [N m s/rad]   -> Researched (for light pendulum low-friciton pivots) 
# b is the pivot's viscous damping coefficient. 
#It models the friction at the joint where the pendulum attaches to the cart, 
#and it makes the pendulum lose energy as it swings - increase in B, oscillation dies faster 

# Derived quantities for sanity checks
w0   = np.sqrt(m * g * l / I)   # natural frequency [rad/s] - doesnt depend on b 
zeta = b / (2 * I * w0)         # damping ratio
# period: 2*pi/w0 
print(f"w0 = {w0:.3f} rad/s (period {2*np.pi/w0:.2f} s), zeta = {zeta:.4f}")

# ---- Cart limits (given): converted from mm to m ----
#recovered in M0
a_max = 2621.03e-3      # max cart acceleration [m/s^2]   -> GIVEN (2621.03 mm/s^2)
v_max = 163.43e-3       # max cart speed [m/s]            -> GIVEN (163.43 mm/s)

# Cart acceleration pulse: rectangular, size a, lasting tau seconds.
# The cart reaches speed a*tau at the end of the pulse, so to respect the speed limit
# even at full acceleration we need tau <= v_max/a_max. We use exactly that value:
tau = v_max / a_max                     # pulse duration [s] (= 0.0624 s, "sharp")
a_values = np.linspace(a_max/12, a_max, 12)   # 12 pulse sizes, from a_max/12 up to a_max [m/s^2]  <-- THE SWEPT VARIABLE

# Sanity check: no pulse may exceed the cart's speed limit
assert np.all(a_values * tau <= v_max + 1e-12), "pulse exceeds cart speed limit"
print(f"pulse duration tau = {tau*1000:.1f} ms, max cart speed reached = {a_values.max()*tau*1000:.2f} mm/s")

t_end  = 6.0                           # simulation length [s]
t_eval = np.linspace(0, t_end, 3000)   # times at which we record the solution

## 2. Cart acceleration input

In [ ]:
#t = current time in sec
#a = acceleration pulse 
#tau = max. time for pulse calc above 
# if time is less than max, time (tau) return the acceleration (x ddot).
# after t, the cart coasts at costnat speed so acc goes to zero. 
#called in lower functions to solve x ddot 
#the sharp acceleration as a rectangular pulse of magnitude a and duration τ, 
#and feed it into the pendulum equation as the input ẍ(t). 
def cart_accel(t, a):
    """Rectangular pulse: acceleration a for 0 <= t < tau, zero afterwards."""
    return a if t < tau else 0.0

## 3. Equations of motion (state = [theta, theta_dot])

In [ ]:
# initial state for s = 0, 0 - starts at bottom and rest
#solver tracks s and passes into function evrytime called
#solve_ivp runs loop, using new : 
#θ_new = θ_old + ω · dt
#ω_new = ω_old + α · dt

def nonlinear_rhs(t, s, a):
    theta, omega = s
    #given acc and time from 12 samples - known acc. max 
    xddot = cart_accel(t, a)
    # I*theta_ddot = -m*l*cos(theta)*xddot - m*g*l*sin(theta) - b*theta_dot
    alpha = (-m*l*np.cos(theta)*xddot - m*g*l*np.sin(theta) - b*omega) / I
    return [omega, alpha]

def linear_rhs(t, s, a):
    theta, omega = s
    xddot = cart_accel(t, a)
    # small-angle version: sin(theta) -> theta, cos(theta) -> 1
    alpha = (-m*l*xddot - m*g*l*theta - b*omega) / I
    return [omega, alpha]
#takes function above and uses a pulse size given to run the simuation, returns theta over time 
def simulate(rhs, a):
    """Integrate from rest at the bottom. max_step stops the solver stepping over the short pulse."""
    #rhs = linear or nonlinear fucniton call above 
    # 0 to t_end = simualte from time 0  - 6s
    # 0, 0 = initial state 
    # Extra arguments passed to rhs after t and s. This is how a reaches cart_accel
#By default solve_ivp uses the RK45 method, a Runge-Kutta scheme. 
#The simplest version of the idea is Euler's method:
#Starting from θ = 0, ω = 0, it repeatedly calls rhs, takes a small step, and repeats. 
#RK45 does this more accurately, by evaluating rhs several times per step and adjusting 
#the step size automatically.
    #max step = allow small ms iterations 
    sol = solve_ivp(rhs, (0, t_end), [0.0, 0.0], args=(a,),
                    t_eval=t_eval, max_step=0.002, rtol=1e-8, atol=1e-10)
    # sol.y is a 2-row array, row 0 is theta t and row 1 is omega t 
    return sol.y[0]   # theta(t)

#simulate gives the solver the starting state [0, 0].
#The solver calls rhs(t, s, a) many times.
#Each call reads θ and ω from s, gets ẍ from cart_accel, computes α from the equation of motion, and returns [ω, α].
#The solver integrates those derivatives forward in time.
#The output is θ(t), the pendulum angle over time.

## 4. Run the sweep

In [ ]:
#create empty list to collect output
results = []

#loops over 12 diff pulse sizes - asked by M1 Milestone
for a in a_values:
    results.append((a, simulate(nonlinear_rhs, a), simulate(linear_rhs, a)))
#runs nonlinear and linear function to return theta, stores in results array. 
#Record the peak angle swing in each
print("a [m/s^2]   peak nonlinear [deg]   peak linear [deg]")
#converts from radians to degrees 
for a, th_nl, th_lin in results:
    print(f"{a:8.3f}   {np.degrees(np.max(np.abs(th_nl))):16.1f}   {np.degrees(np.max(np.abs(th_lin))):14.1f}")

## 5. Plots

In [ ]:
cmap = plt.cm.viridis(np.linspace(0, 1, len(a_values)))
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

# (a) nonlinear response for every pulse size
for col, (a, th_nl, _) in zip(cmap, results):
    ax[0].plot(t_eval, np.degrees(th_nl), color=col, label=f"a = {a:.2f}")
ax[0].set_title("Nonlinear response, 12 pulse sizes")
ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("theta [deg]")
ax[0].legend(title="cart accel [m/s$^2$]", fontsize=7, ncol=2); ax[0].grid(alpha=0.3)

# (b) nonlinear vs linear for the smallest and largest pulse
for idx in (0, len(a_values) - 1):
    a, th_nl, th_lin = results[idx]
    ax[1].plot(t_eval, np.degrees(th_nl), label=f"nonlinear, a={a:.2f}")
    ax[1].plot(t_eval, np.degrees(th_lin), "--", label=f"linear, a={a:.2f}")
ax[1].set_title("Linearization check: small vs large pulse")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("theta [deg]")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

# (c) peak angle vs pulse size: where does the linearization break down?
peak_nl  = [np.degrees(np.max(np.abs(r[1]))) for r in results]
peak_lin = [np.degrees(np.max(np.abs(r[2]))) for r in results]
ax[2].plot(a_values, peak_lin, "o--", label="linear")
ax[2].plot(a_values, peak_nl,  "s-",  label="nonlinear")
ax[2].set_title("Peak swing angle vs pulse size")
ax[2].set_xlabel("cart acceleration a [m/s$^2$]"); ax[2].set_ylabel("peak |theta| [deg]")
ax[2].legend(); ax[2].grid(alpha=0.3)

fig.suptitle(f"Hanging pendulum on cart | w0={w0:.2f} rad/s, zeta={zeta:.3f}, pulse tau={tau*1000:.0f} ms")
plt.tight_layout()
plt.savefig("pendulum_cart_sweep.png", dpi=150)
plt.show()

# image 1 = every curve starts at 0° and swings negative first. 
#The cart accelerates forward and the pendulum lags behind it, 
#so it swings backward relative to the cart. The first extreme comes at about 0.3 s, 
#roughly a quarter of the 1.16 s period.

#image 2: 

In [ ]:
# ---- Range of target distances (EDIT to match the assignment) ----
# The task: the cart must end up exactly D mm from where it started. We use
# the fastest movement the hardware allows:
#   1. accelerate at the maximum acceleration a_max
#   2. cruise at constant speed once we hit the maximum speed v_max
#   3. decelerate at -a_max so the cart stops at exactly distance D
# Then we measure how big the pendulum's oscillation is AFTER the cart stops.
D_min_mm, D_max_mm, n_D = 5.0, 250.0, 13          # 13 values >= the required 10
D_values_mm = np.linspace(D_min_mm, D_max_mm, n_D)  # target distances [mm]

def motion_profile(D):
    """Timing of the accelerate / cruise / decelerate movement for target distance D [m].
        Returns a dictionary with:
      v_peak   - highest cart speed reached
      t_acc    - how long the acceleration phase lasts (the decel phase lasts the same)
      t_cruise - how long the cart cruises at constant speed (0 for short moves)
      T        - total movement time
    """
    # Distance the cart covers just by speeding up to v_max and slowing back down.
    # Speeding up: distance = v_max^2 / (2*a_max). Slowing down: the same. 
    d_ramp = v_max**2 / a_max          # distance used by accelerating + decelerating to/from v_max
    if D >= d_ramp:                    # trapezoid: reaches v_max and cruises
        # LONG MOVE ("trapezoid" speed profile): there is room to reach v_max,
        # so the cart accelerates, cruises at v_max, then decelerates.
        v_peak   = v_max
        t_acc    = v_max / a_max
        t_cruise = (D - d_ramp) / v_max
    else:                              # triangle: D = a*t^2  ->  never reaches v_max\
        # SHORT MOVE ("triangle" speed profile): the cart runs out of distance
        # before it reaches v_max. It accelerates for time t, then immediately
        # decelerates for time t. Distance = a*t^2 - acc and deacc distance,
        # so t = sqrt(D/a). v = a*t = sqrt(a*D). 
        v_peak   = np.sqrt(a_max * D)
        t_acc    = v_peak / a_max
        t_cruise = 0.0
    T = 2*t_acc + t_cruise             # total movement time [s]
    # Sanity check:  the distance travelled is the area under the speed-vs-time
    # curve. Two triangles (accel and decel) give v_peak*t_acc in total, and the
    # cruise rectangle gives v_peak*t_cruise. This must equal the requested D.
    assert abs(v_peak*(t_acc + t_cruise) - D) < 1e-9, "profile does not cover distance D"
    return dict(D=D, v_peak=v_peak, t_acc=t_acc, t_cruise=t_cruise, T=T)

def cart_accel_D(t, p):
    """Cart acceleration xddot(t): +a_max, then 0 (cruise), then -a_max, then 0 (stopped)."""
    if t < p['t_acc']: # phase 1: speeding up
        return  a_max
    elif t < p['t_acc'] + p['t_cruise']:# phase 2: cruising, no acceleration
        return  0.0
    elif t < p['T']:  # phase 3: slowing down (negative acceleration)
        return -a_max
    else:
        return  0.0  # phase 4: cart has stopped
def cart_velocity(t, p):
    """Cart speed at time t (used only for plotting)."""
    ta, tc, T = p['t_acc'], p['t_cruise'], p['T']
    if t < ta:        return a_max * t
    elif t < ta + tc: return p['v_peak']
    elif t < T:       return p['v_peak'] - a_max * (t - ta - tc)
    else:             return 0.0

In [ ]:
from scipy.integrate import trapezoid

# p is a small "movement profile": a dictionary that stores the timing of ONE
# cart movement for ONE target distance D. It holds five numbers:
#     p['D']        target distance [m]
#     p['v_peak']   highest speed reached [m/s]
#     p['t_acc']    how long the speed-up phase lasts (the slow-down lasts the same) [s]
#     p['t_cruise'] how long the cart cruises at constant speed [s]
#     p['T']        total movement time [s]
#
# It is created once per run, in simulate_distance(), by p = motion_profile(D).
# It is then passed along to every function that needs to know WHEN the cart
# speeds up, cruises, and slows down (cart_accel_D, pendulum_rhs_D, etc.).
# Each of those functions receives it under the name p. It is the same
# information, handed along, not recalculated.
# A new D gives a new p, because the timing changes with the distance.

def pendulum_rhs_D(t, s, p, linear=False):
    """Same equation of motion of pendulum."""
    theta, omega = s
    xddot = cart_accel_D(t, p)  # cart acceleration at this instant
    if linear:   # small-angle model
        alpha = (-m*l*xddot - m*g*l*theta - b*omega) / I
    else:        # full nonlinear model
        alpha = (-m*l*np.cos(theta)*xddot - m*g*l*np.sin(theta) - b*omega) / I
    return [omega, alpha]

def simulate_distance(D, linear=False, t_settle=6.0):
    """Simulate the cart moving distance D [m]; keep t_settle seconds after it stops.Returns 
    (profile, time array, theta array)"""
    p = motion_profile(D)
    t_end_D = p['T'] + t_settle
    t = np.linspace(0, t_end_D, int(t_end_D/0.002) + 1)
    sol = solve_ivp(pendulum_rhs_D, (0, t_end_D), [0.0, 0.0], args=(p, linear),
                    t_eval=t, max_step=0.002, rtol=1e-8, atol=1e-10)
    return p, sol.t, sol.y[0]      # profile, time, theta(t)

def analytic_amplitude(p):
    """Linear undamped prediction of the leftover amplitude after the cart stops [rad].
    The integral measures how well the cart's acceleration is 'in tune' with the
    pendulum's natural frequency w0."""
    tt = np.linspace(0, p['T'], 20001)
    ax = np.array([cart_accel_D(ti, p) for ti in tt])
    integral = trapezoid(ax*np.exp(1j*w0*tt), tt)
    return m*l/(I*w0) * abs(integral)

def residual_amplitude(t, theta, p):
    """Peak |theta| after the cart has stopped (t >= T) [rad]."""
    return np.max(np.abs(theta[t >= p['T']]))

In [ ]:
sweep = []
#loops yhrough all D values adn holds in sweep 
for D_mm in D_values_mm:
    D = D_mm / 1000.0                                  # mm -> m
    # 2 simulations, simulate_distance returns three things: the movement profile p, 
    #the time array t, and the pendulum angle array theta.Called linear (true) and non linear (false)
    p, t, th_nl  = simulate_distance(D, linear=False)
    _, _, th_lin = simulate_distance(D, linear=True)
    sweep.append(dict(D_mm=D_mm, p=p, t=t, th_nl=th_nl, th_lin=th_lin,
                      A_nl=residual_amplitude(t, th_nl, p),
                      A_lin=residual_amplitude(t, th_lin, p),
                      A_an=analytic_amplitude(p)))
#bundles in sweep: 
#d_mm = distance in mm, p = movement profile, t = time array, 
#th_nl/th_lin = angle vs time curves, A_lin/A-an = linear model prediction, Ideal model prediction (no damping)
print(f"Triangle/trapezoid boundary: D = v_max^2/a_max = {1000*v_max**2/a_max:.1f} mm")
print(" D [mm]  v_peak [mm/s]  T [s]  A nonlinear [deg]  A linear [deg]  A analytic [deg]")
for r in sweep:
    print(f"{r['D_mm']:7.1f}  {1000*r['p']['v_peak']:12.1f}  {r['p']['T']:5.2f}"
          f"  {np.degrees(r['A_nl']):16.2f}  {np.degrees(r['A_lin']):14.2f}  {np.degrees(r['A_an']):15.2f}")

In [ ]:
from matplotlib.colors import Normalize

norm = Normalize(D_min_mm, D_max_mm)
colors = plt.cm.viridis(norm(D_values_mm))
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
# (a) cart speed vs time: shows the movement pattern (triangle -> trapezoid)
for col, r in zip(colors, sweep):
    tv = np.linspace(0, r['p']['T'] + 0.2, 600)
    ax[0].plot(tv, [1000*cart_velocity(ti, r['p']) for ti in tv], color=col)
ax[0].axhline(1000*v_max, color='k', ls=':', lw=1, label='v_max')
ax[0].set_title("Cart speed profile for each distance D")
ax[0].set_xlabel("time [s]"); ax[0].set_ylabel("cart speed [mm/s]"); ax[0].legend(); ax[0].grid(alpha=0.3)

# (b) pendulum angle vs time for each D (nonlinear model)
for col, r in zip(colors, sweep):
    ax[1].plot(r['t'], np.degrees(r['th_nl']), color=col, lw=0.9)
ax[1].set_title("Pendulum angle after the movement (nonlinear)")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("theta [deg]"); ax[1].grid(alpha=0.3)
sm = plt.cm.ScalarMappable(cmap='viridis', norm=norm)
fig.colorbar(sm, ax=ax[1], label="target distance D [mm]", pad=0.02)

# (c) residual oscillation amplitude vs D
ax[2].plot(D_values_mm, [np.degrees(r['A_nl']) for r in sweep], 'o-', label='simulation, nonlinear')
ax[2].plot(D_values_mm, [np.degrees(r['A_lin']) for r in sweep], 's--', label='simulation, linear')
Dfine = np.linspace(D_min_mm, D_max_mm, 300)
ax[2].plot(Dfine, [np.degrees(analytic_amplitude(motion_profile(d/1000))) for d in Dfine],
           'k:', label='analytic (linear, undamped)')
ax[2].axvline(1000*v_max**2/a_max, color='gray', lw=0.8, ls='--')
ax[2].text(1000*v_max**2/a_max + 4, 5, 'D = v_max^2/a_max', rotation=90, fontsize=8, color='gray')
ax[2].set_title("Oscillation amplitude after stopping vs distance D")
ax[2].set_xlabel("target distance D [mm]"); ax[2].set_ylabel("peak |theta| after stop [deg]")
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("pendulum_distance_sweep.png", dpi=150, bbox_inches='tight')
plt.show()